In [1]:
from _utils import *

def summarize_predictions(row):
    preds = [row["currents1"], row["currents2"], row["currents3"]]

    vc = pd.Series(preds).value_counts()
    #note: idxmax returns whose label value is the largest (rather than the largest index)
    #in a tie (if all counts are 1, and there is no majority vote), it will choose currents1
    label_pred = vc.idxmax()
    count = vc.max()

    # confidence mapping
    if count == 1:
        confidence = "low"
    elif count == 2:
        confidence = "medium"
    else:
        confidence = "high"

    return pd.Series({
        "file_hash": row["file_hash"],
        "gpt_pred": label_pred,
        "gpt_confidence": confidence,
        "votes": count,
        "agreement": count / 3
    })


def adjust_gpt_pred(row):
    pred = row['gpt_pred']
    count = row['gpt_pred_count']
    
    if count > 1:
        # Take the first element
        return pred[0] if isinstance(pred, list) else pred
    elif count == 0:
        # Empty list case
        return "Z Neither"
    else:  # count == 1
        # Remove brackets, return the single element
        return pred[0] if isinstance(pred, list) else pred

def map_gpt_subtype(subtype):
    if subtype in ['Z Neither','I Mixed','Na+-glutamate transporter','I Cl, leak','I Chloride']:
        return 'Z Neither'
    elif subtype in ['I Na,t']:
        return "I Na (Transient)"
    elif subtype in ['I A','I A, slow']:
        return 'I K (A-type)'
    elif subtype in ['I K']:
        return 'I K (Delayed Rectifier)'
    elif subtype in ['I Calcium']:
        return 'I Ca (Rare)'
    elif subtype in ['I L high threshold','I R','I Ca,p','I p,q','I Q','I N']:
        return 'I Ca (HVA)'
    elif subtype in ['AMPA']:
        return 'R Glutamate (AMPA)'
    elif subtype in ['I h']:
        return 'I H'
    elif subtype in ['I M','KCNQ1']:
        return 'I K (M-type)'
    elif subtype in ['I T low threshold']:
        return 'I Ca (T-type LT)'
    elif subtype in ['I Sodium','I Na, leak']:
        return 'I Na (Rare)'
    elif subtype in ['Ca pump','I SERCA','I Calcium']:
        return 'I Ca (Rare)'
    elif subtype in ['NMDA']:
        return 'R Glutamate (NMDA)'
    elif subtype in ['GabaA','Gaba','GabaB']:
        return 'R GABA'
    elif subtype in ['I Potassium','Kir2 leak','I_HERG','I Kur','ATP-senstive potassium current','I K,leak','I Ks','I_KD','I_KHT','I_KLT','I_Ks','Kir','Kir, inactivating']:
        return 'I K (Rare)'
    elif subtype in ['I Na,p']:
        return 'I Na (Persistent)'
    elif subtype in ['I_K,Ca','IK Skca','IK Bkca','I C','I_AHP','I K,Ca']:
        return 'I K (Ca-activated)'
    elif subtype in ['Glutamate','Alpha','Nicotinic','mGluR','Sensory Receptors','P2X3','Glycine']:
        return 'R Other (Rare)'
    elif subtype in ['Channelrhodopsin (ChR)','I CAN']:
        return 'I Other (Rare)'
    elif subtype in ['I Na, slow inactivation']:
        return 'I Na (Slow inactivation)'
    else:
        return 'Unknown'

def map_gpt_type(subtype):
    if subtype in ['Z Neither','I H']:
        return subtype
    elif subtype in ['I Na (Transient)','I Na (Persistent)','I Na (Slow inactivation)','I Na (Rare)']:
        return 'I Na'
    elif subtype in ['I Ca (Rare)','I Ca (HVA)','I Ca (T-type LT)']:
        return 'I Ca'
    elif subtype in ['I K (Rare)','I K (A-type)','I K (Ca-activated)','I K (Delayed Rectifier)','I K (M-type)']:
        return 'I K'
    elif subtype in ['R Glutamate (AMPA)', 'R Glutamate (NMDA)','R GABA']:
        return 'Receptor'
    elif subtype in ['R Other (Rare)']:
        return 'R Other'
    elif subtype in ['I Other (Rare)']:
        return 'I Other'
    else:
        return 'Unknown'

In [2]:
raw_gpt_df = pd.read_csv(RAW_DATA_DIR / "mod_files_gpt.csv").rename(columns={"hash":"file_hash"})
ant_df = pd.read_csv(PIPELINE_DATA_DIR / "anno_with_split.csv")
ANNOTATED_SAMPLES = ant_df["file_hash"].tolist()

In [3]:
raw_gpt_df.columns

Index(['file_hash', 'currents1', 'notes1', 'currents2', 'notes2', 'currents3',
       'notes3', 'currents_at_least_1', 'currents_at_least_2',
       'currents_at_least_3'],
      dtype='object')

In [4]:
log = DataLogger()
log.add_entry("raw gpt file", raw_gpt_df)
log.get_log()

,step,n_row,n_hash
0,raw gpt file,5133,5133


In [5]:
summary_df = raw_gpt_df.apply(summarize_predictions, axis=1)

In [6]:
raw_gpt_df[["currents1","currents2","currents3"]].head()

,currents1,currents2,currents3
0,"[""IK Skca""]","[""IK Skca""]","[""IK Skca""]"
1,"[""I Sodium"", ""I Potassium"", ""I Calcium"", ""NMDA""]","[""I Potassium"", ""I Sodium"", ""I Calcium"", ""NMDA""]","[""I Potassium"", ""I Sodium"", ""I Calcium"", ""NMDA""]"
2,"[""I T low threshold""]","[""I T low threshold""]","[""I T low threshold""]"
3,"[""I K""]","[""I K""]","[""I K""]"
4,"[""I Sodium""]","[""I Na,t""]","[""I Na,t""]"


In [7]:
View(summary_df.head())

,file_hash,gpt_pred,gpt_confidence,votes,agreement
0,cf53b079807fc6711d9563031aae227dd1a73cf5add973fd249463a3f2cf8fba,"[""IK Skca""]",high,3,1.000000
1,04c32cd101754ad7f2eaf278919c6ce85b8a7d67149fca0bb1c883cb9f45abcc,"[""I Potassium"", ""I Sodium"", ""I Calcium"", ""NMDA""]",medium,2,0.666667
2,2d3c5c121271cfad6d2e35db7d2c418de64f97e57c20a0f0ddf9da411794118e,"[""I T low threshold""]",high,3,1.000000
3,e2f71ea16f8378b3cf0670f9bd5e457634824ab9ee8a7c000f891dce1b05412a,"[""I K""]",high,3,1.000000
4,04860f65d331d381e779e488566f95ab0e1fa4960b50a768628fe980ea8744db,"[""I Na,t""]",medium,2,0.666667


In [8]:
gpt_df2 = summary_df[["file_hash","gpt_pred","gpt_confidence"]]
log.add_entry("add confidence (no row change exp)", gpt_df2)
log.get_log()

,step,n_row,n_hash
0,raw gpt file,5133,5133
1,add confidence (no row change exp),5133,5133


In [9]:
gpt_df3 = gpt_df2.query("file_hash in @ANNOTATED_SAMPLES")
log.add_entry("filter to annotated samples", gpt_df3)
log.get_log()

,step,n_row,n_hash
0,raw gpt file,5133,5133
1,add confidence (no row change exp),5133,5133
2,filter to annotated samples,1023,1023


In [10]:
gpt_df4 = gpt_df3.copy()
gpt_df4['gpt_pred'] = gpt_df4['gpt_pred'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
gpt_df4['gpt_pred_count'] = gpt_df4['gpt_pred'].apply(lambda x: len(x) if isinstance(x, list) else 0)
gpt_df4['gpt_pred_adj'] = gpt_df4.apply(adjust_gpt_pred, axis=1)
gpt_df4['gpt_pred_subtype_label'] = gpt_df4['gpt_pred_adj'].apply(map_gpt_subtype)
gpt_df4['gpt_pred_type_label'] = gpt_df4['gpt_pred_subtype_label'].apply(map_gpt_type)

In [11]:
groups = ["gpt_pred_subtype_label","gpt_pred_adj"]
gpt_df4[groups].groupby(groups).count()

Empty DataFrame
Columns: []
Index: [(I Ca (HVA), I Ca,p), (I Ca (HVA), I L high threshold), (I Ca (HVA), I N), (I Ca (HVA), I Q), (I Ca (HVA), I R), (I Ca (HVA), I p,q), (I Ca (Rare), Ca pump), (I Ca (Rare), I Calcium), (I Ca (Rare), I SERCA), (I Ca (T-type LT), I T low threshold), (I H, I h), (I K (A-type), I A), (I K (A-type), I A, slow), (I K (Ca-activated), I C), (I K (Ca-activated), I K,Ca), (I K (Ca-activated), IK Bkca), (I K (Ca-activated), IK Skca), (I K (Ca-activated), I_AHP), (I K (Ca-activated), I_K,Ca), (I K (Delayed Rectifier), I K), (I K (M-type), I M), (I K (M-type), KCNQ1), (I K (Rare), ATP-senstive potassium current), (I K (Rare), I K,leak), (I K (Rare), I Ks), (I K (Rare), I Kur), (I K (Rare), I Potassium), (I K (Rare), I_HERG), (I K (Rare), I_KD), (I K (Rare), I_KHT), (I K (Rare), I_KLT), (I K (Rare), I_Ks), (I K (Rare), Kir), (I K (Rare), Kir, inactivating), (I K (Rare), Kir2 leak), (I Na (Persistent), I Na,p), (I Na (Rare), I Na, leak), (I Na (Rare), I Sodium), (I Na (Slow inactivation), I Na, slow inactivation), (I Na (Transient), I Na,t), (I Other (Rare), Channelrhodopsin (ChR)), (I Other (Rare), I CAN), (R GABA, Gaba), (R GABA, GabaA), (R GABA, GabaB), (R Glutamate (AMPA), AMPA), (R Glutamate (NMDA), NMDA), (R Other (Rare), Glutamate), (R Other (Rare), Glycine), (R Other (Rare), Sensory Receptors), (Unknown, KCC2), (Unknown, Na/Ca exchanger), (Unknown, Na/K pump), (Z Neither, I Chloride), (Z Neither, I Mixed), (Z Neither, Na+-glutamate transporter), (Z Neither, Z Neither)]

In [12]:
groups = ["gpt_pred_type_label","gpt_pred_subtype_label"]
gpt_df4[groups].groupby(groups).count()

Empty DataFrame
Columns: []
Index: [(I Ca, I Ca (HVA)), (I Ca, I Ca (Rare)), (I Ca, I Ca (T-type LT)), (I H, I H), (I K, I K (A-type)), (I K, I K (Ca-activated)), (I K, I K (Delayed Rectifier)), (I K, I K (M-type)), (I K, I K (Rare)), (I Na, I Na (Persistent)), (I Na, I Na (Rare)), (I Na, I Na (Slow inactivation)), (I Na, I Na (Transient)), (I Other, I Other (Rare)), (R Other, R Other (Rare)), (Receptor, R GABA), (Receptor, R Glutamate (AMPA)), (Receptor, R Glutamate (NMDA)), (Unknown, Unknown), (Z Neither, Z Neither)]

In [13]:
gpt_df5 = gpt_df4.copy()
gpt_df5 = gpt_df4[["file_hash","gpt_pred_subtype_label","gpt_pred_type_label","gpt_confidence"]]

In [14]:
gpt_df5.to_csv(PIPELINE_DATA_DIR  /"gpt_pred.csv", index=False)